# 18b96e40 flywheel iteration 1 — aggregate selfplay + crisis

This is the primary **play → mine → train** arm.  It warm-starts the current
18-block × 96-channel actor at epoch 40 and trains on every unique, same-actor
search trajectory generated in iteration 1:

- 2,945,600 clean exploit-search states;
- 723,704 explore states with behavior noise separated from clean labels;
- 1,140,229 recovery/prevention crisis states from two disjoint, independently
  causally validated tranches;
- 1,798,692 actor-native greedy states for frozen-policy KL preservation.

The target tensor has **4,809,533 unique states**.  Pilot crisis is included
once; the expansion adds only its new seeds.  No vh3, 192/256-channel, 3f, or 3k
games enter this run.

The whole-trajectory objective is intentionally different from the sparse-edit
diagnostic: hard clean-search CE on all search states plus frozen-base KL on all
anchors.  Crisis sources receive 2× loss weight, so broad board diversity is
kept while crisis is not diluted.  BatchNorm is frozen, policy forwards are
FP16, losses are FP32, batches are pooled, legality is exact, color augmentation
is enabled, and D4 is held out of this first arm.  Twelve epochs at batch 4096
give about 18k optimizer updates—comparable to the successful 40-epoch
from-scratch update count—while epoch checkpoints expose the entire trajectory.

Upload these files to `MyDrive/alphatrain/`:

- `colorlines_pillar3d_v5.tar.gz` — 1,320,947 bytes, SHA256 `e07f600bb1243d8b4d5263eee6a7d8f5add95e2c8479c16024512bb00e7ade7d`
- `flywheel_18b96e40_i1_mixed_targets.pt.gz` — 699,323,866 bytes, SHA256 `4c58540da1190b5cb4b3f7d3cc61dbbe5fb4e5c7436452f02cdbf3d11d674762`
- `flywheel_18b96e40_i1_anchors.pt.gz` — 42,777,441 bytes, SHA256 `76729fa980be36667676b6012e3bd9de2904f45854cbc99c41be3dbcd03a837f`
- `scratch18b96_lr3e3_ckpts_epoch_40.pt` — 36,595,064 bytes, SHA256 `233809b150e94341b250d31a112111fc36381f5f9604c6d67afcaa738740da93`


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import gc, hashlib, os, shutil, time

DRIVE = '/content/drive/MyDrive/alphatrain'
ARTIFACTS = {
  "colorlines_pillar3d_v5.tar.gz": {
    "size": 1320947,
    "sha256": "e07f600bb1243d8b4d5263eee6a7d8f5add95e2c8479c16024512bb00e7ade7d"
  },
  "flywheel_18b96e40_i1_mixed_targets.pt.gz": {
    "size": 699323866,
    "sha256": "4c58540da1190b5cb4b3f7d3cc61dbbe5fb4e5c7436452f02cdbf3d11d674762"
  },
  "flywheel_18b96e40_i1_anchors.pt.gz": {
    "size": 42777441,
    "sha256": "76729fa980be36667676b6012e3bd9de2904f45854cbc99c41be3dbcd03a837f"
  },
  "scratch18b96_lr3e3_ckpts_epoch_40.pt": {
    "size": 36595064,
    "sha256": "233809b150e94341b250d31a112111fc36381f5f9604c6d67afcaa738740da93"
  }
}

def sha256(path, chunk=8 << 20):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        while True:
            block = f.read(chunk)
            if not block:
                return h.hexdigest()
            h.update(block)

for name, expected in ARTIFACTS.items():
    src = os.path.join(DRIVE, name)
    dst = os.path.join('/content', name)
    assert os.path.exists(src), f'missing Drive upload: {src}'
    shutil.copy2(src, dst)
    size = os.path.getsize(dst)
    digest = sha256(dst)
    assert size == expected['size'], (name, size, expected['size'])
    assert digest == expected['sha256'], (name, digest)
    print(f'verified {name}: {size:,} bytes')

os.makedirs('/content/alphatrain/data', exist_ok=True)
shutil.unpack_archive('/content/colorlines_pillar3d_v5.tar.gz', '/content')

for gz_name, pt_name, pt_size, pt_sha in [
    ('flywheel_18b96e40_i1_mixed_targets.pt.gz', 'flywheel_18b96e40_i1_mixed_targets.pt', 2467307383,
     '6023a35320e743c3ba52535e2f1313d5486ab029648ba6a0f3c58076d9f57d8e'),
    ('flywheel_18b96e40_i1_anchors.pt.gz', 'flywheel_18b96e40_i1_anchors.pt', 296797651,
     '45960a3d743a51be32f9d2a2431f32b944f4e40be5a10ec7d2a1fc753790a3fa'),
]:
    src = os.path.join('/content', gz_name)
    dst = os.path.join('/content/alphatrain/data', pt_name)
    t0 = time.time()
    import gzip
    with gzip.open(src, 'rb') as fin, open(dst, 'wb') as fout:
        shutil.copyfileobj(fin, fout, length=8 << 20)
    assert os.path.getsize(dst) == pt_size, (pt_name, os.path.getsize(dst))
    assert sha256(dst) == pt_sha, f'decompressed SHA256 mismatch: {pt_name}'
    os.remove(src)
    print(f'unpacked {pt_name}: {pt_size:,} bytes in {time.time()-t0:.0f}s')

shutil.copy2('/content/scratch18b96_lr3e3_ckpts_epoch_40.pt', '/content/alphatrain/data/scratch18b96_lr3e3_ckpts_epoch_40.pt')

source = open('/content/alphatrain/train_flywheel.py').read()
for marker in ('class PooledBatchLoader', '--resume-every-steps',
               '_quantize_frozen_bn_for_deployment_', '--source-weights'):
    assert marker in source, f'STALE CODE TARBALL: missing {marker}'
print('Flywheel pooled/resumable/frozen-BN trainer confirmed.')


In [ ]:
import torch
assert torch.cuda.is_available(), 'Select a CUDA Colab runtime'
gpu = torch.cuda.get_device_properties(0)
print(f'PyTorch {torch.__version__} | {torch.cuda.get_device_name(0)} | '
      f'{gpu.total_memory/1e9:.1f} GB')


In [ ]:
# ===== PRIMARY AGGREGATE FLYWHEEL CONFIG =====
RUN = '18b96e40_i1_mixed_warm'
EPOCHS = 12
BATCH = 4096
LR = 1e-4
SOURCE_WEIGHTS = [1.0, 1.0, 2.0, 2.0]  # exploit, explore, old crisis, new crisis
ANCHOR_WEIGHT = 1.0
SEED = 20260818
SAVE_STEPS = 250     # model-only dose checkpoints within every full epoch
RESUME_STEPS = 1000  # overwrite crash-safe latest.pt within long epochs
print(RUN, 'epochs=', EPOCHS, 'batch=', BATCH, 'lr=', LR,
      'source_weights=', SOURCE_WEIGHTS)


In [ ]:
# Cheap metadata/provenance preflight; tensors stay memory-mapped.
import torch

target_path = '/content/alphatrain/data/flywheel_18b96e40_i1_mixed_targets.pt'
anchor_path = '/content/alphatrain/data/flywheel_18b96e40_i1_anchors.pt'
target = torch.load(target_path, map_location='cpu', weights_only=False, mmap=True)
anchor = torch.load(anchor_path, map_location='cpu', weights_only=False, mmap=True)

assert len(target['boards']) == 4809533
assert len(anchor['boards']) == 1798692
assert target['metadata']['source_names'] == ['18b96e40_i1_exploit_400', '18b96e40_i1_explore_200x400', '18b96e40_i1_crisis_pilot_600_1600', '18b96e40_i1_crisis_expand_600_1600']
assert target['metadata']['lineage'] == 'scratch18b96_lr3e3_e40'
assert anchor['metadata']['lineage'] == 'scratch18b96_lr3e3_e40'
assert target['metadata']['base_checkpoint_sha256'] ==        '233809b150e94341b250d31a112111fc36381f5f9604c6d67afcaa738740da93'
assert target['metadata']['split_semantics'] == 'group_seed_mod_20; 1=validation'
print('target rows:', len(target['boards']), target['metadata']['source_names'])
print('anchor rows:', len(anchor['boards']))
print('validation:', int((target['split'] == 1).sum()), 'target rows;',
      int((anchor['split'] == 1).sum()), 'anchor rows')
del target, anchor
gc.collect()


In [ ]:
# Resume exactly from Drive after a Colab interruption.
SAVE_DIR = f'{DRIVE}/{RUN}_ckpts'
LATEST = f'{SAVE_DIR}/latest.pt'
RESUME_ARGS = f'--resume {LATEST}' if os.path.exists(LATEST) else ''
SOURCE_WEIGHT_ARGS = ' '.join(str(x) for x in SOURCE_WEIGHTS)
print('Resume:', LATEST if RESUME_ARGS else 'none (new warm-start run)')

%cd /content
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python -m alphatrain.train_flywheel     --targets alphatrain/data/flywheel_18b96e40_i1_mixed_targets.pt     --anchors alphatrain/data/flywheel_18b96e40_i1_anchors.pt     --base alphatrain/data/scratch18b96_lr3e3_ckpts_epoch_40.pt     {RESUME_ARGS}     --objective aggregate --soft-alpha 0     --source-weights {SOURCE_WEIGHT_ARGS} --anchor-weight {ANCHOR_WEIGHT}     --epochs {EPOCHS} --batch-size {BATCH} --lr {LR} --weight-decay 0     --schedule cosine --warmup-fraction 0.05 --min-lr-ratio 0.05     --device cuda --precision fp16 --legal-support-loss     --no-dihedral-augment --color-augment --augment-factor 1     --save-every-steps {SAVE_STEPS} --resume-every-steps {RESUME_STEPS}     --archive-every-epochs 1 --audit-batches 10 --log-every 25     --gate-min-retain 0.90 --gate-max-legal-kl 0.05     --gate-max-new-bn-nonfinite 0     --seed {SEED} --save-dir {SAVE_DIR} 2>&1 | tee /content/{RUN}_train.log


In [ ]:
# Inspect the resumable state and per-epoch functional diagnostics.
import glob, json, os

for path in sorted(glob.glob(f'{SAVE_DIR}/diagnostics_epoch_*.json')):
    report = json.load(open(path))
    target = report['audit']['target']
    anchor = report['audit']['anchor']
    print(os.path.basename(path),
          f"step={report['global_step']}",
          f"target retain={target['retain']:.4f} KL={target['mean_legal_kl']:.6f}",
          f"anchor retain={anchor['retain']:.4f} KL={anchor['mean_legal_kl']:.6f}",
          'gate=', report['functional_gate']['failures'])

print('Model archives:')
for path in sorted(glob.glob(f'{SAVE_DIR}/epoch_*.pt')):
    print(path, f'{os.path.getsize(path)/1e6:.1f} MB')


## Local M5 promotion gate

Download epochs 1, 2, 3, 5, 8, and 12 first.  Export each checkpoint directly
to its own TorchScript path, then use aggregate-only MPS FP16 evaluation.  A
999-game run is only a catastrophe/trajectory screen; do not interpret
individual seeds or rank close candidates from it.

```bash
cd /path/to/colorlines98
PYTHONPATH=. NUMBA_CACHE_DIR=/tmp/colorlines98-numba-cache   caffeinate -i -s .venv/bin/python -m alphatrain.inference_cpp.export_ts   --model alphatrain/data/18b96e40_i1_mixed_warm_epoch_3.pt   --output alphatrain/inference_cpp/data/18b96e40_i1_mixed_warm_epoch_3_ts.pt

cd alphatrain/inference_cpp
caffeinate -i -s ./build/eval   --model data/18b96e40_i1_mixed_warm_epoch_3_ts.pt --device mps --batch 500   --seed-start 1000000 --seed-end 1000999 --max-turns 1000   --scores-out data/18b96e40_i1_mixed_warm_epoch_3_cap1k_1000k_999.csv
```

Take at most two trajectory survivors to the full 5,000-game development
distribution (`--seed-end 1005000`) and evaluate the untouched epoch-40 base on
that distribution once.  Compare them with independent resampling, never
per-seed deltas:

```bash
cd /path/to/colorlines98
.venv/bin/python -m alphatrain.scripts.compare_score_distributions   alphatrain/inference_cpp/data/18b96e40_base_cap1k_1000k_5k.csv   alphatrain/inference_cpp/data/18b96e40_i1_mixed_warm_epoch_3_cap1k_1000k_5k.csv
```

Promotion requires a distributional improvement without a material floor or
cap-rate regression.  Matching numeric seed IDs are environmental controls,
not paired stochastic-game evidence.  The reserved final bank
1,100,000–1,119,999 remains untouched until a development winner exists.
